# UDL Assignment 2 — Part A
## Beta-VAE and VQ-VAE with PixelCNN prior
**Group 56** · **Ankit Kishor Bukshete** · **2025AE05016**  
Add the remaining members of your Assignment 1 group to `config.json` before submission.

**Status:** implementation prepared; full CIFAR-10 training and metrics are not yet run. This notebook and HTML must be executed/exported on the mandatory BITS Prayogshala infrastructure. The current HTML is a source preview, not a completed submission.

## 0. Read before running
1. Upload/extract the entire project in your BITS GPU workspace. Install dependencies in the selected kernel if needed.
2. Confirm BITS lab usage in `config.json`, add all group members, and keep `SMOKE=False` for the final run.
3. Allow CIFAR-10 and the official Inception weights to download. A shared 45k/5k/10k train/validation/test protocol is used.
4. Checkpoints are saved after each epoch. Run the same cells to resume. If you change training settings, use a new `run_dir`.
5. Save this notebook with its outputs. Complete Part B, observations and the required JPG before finalizing.

The configured 100 autoencoder epochs / 50 prior epochs are a starting training budget, not a guarantee of good metrics. Check PSNR/loss trends, code utilization, prior cross-entropy and generated examples. If results are weak or still improving, document a new training run rather than submit inadequate training.

In [ ]:
from pathlib import Path
import json, sys, platform
assert Path("udl/models.py").is_file(), "Open this notebook from the extracted UDL_Assignment2 project folder."
import torch
print("Python:", platform.python_version(), "PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))
# If imports fail: run %pip install -r requirements.txt in this kernel and restart.
# Keep the lab's CUDA-compatible torch/torchvision pair when it already works.

In [ ]:
from udl.pipeline import load_config
SMOKE = False  # True only for a small pipeline check; cannot produce final submission metrics.
# First confirm you are in BITS Prayogshala, then set lab_confirmed=true in config.json.
cfg = load_config("config.json", smoke=SMOKE)
print(json.dumps(cfg, indent=2))

## 1. CIFAR-10 and evaluation protocol
Use `ToTensor()` to scale RGB pixels into [0,1], with no class-label conditioning and no augmentation. A fixed seed-42 partition holds out 5,000 training images for validation and all 10,000 test images for final metrics. Validation PSNR is monitored each epoch; autoencoders and GANs use final checkpoints; PixelCNN uses its best validation-loss checkpoint, without selection on test FID.

FID is computed using 2,048-dimensional Inception features and 10,000 real/10,000 generated images for every model. The library handles resizing internally. VAE generation uses z~N(0,I); VQ-VAE generation uses ancestral PixelCNN codes. Reconstructions and uniformly random codebook entries are not used as generated FID samples.

In [ ]:
from udl.pipeline import prepare_data, record_environment
from IPython.display import display, Image
import pandas as pd
record_environment(cfg)
train, validation, test = prepare_data(cfg)
x, label = train[0]
print("Split sizes:", len(train), len(validation), len(test))
print("Image shape:", tuple(x.shape), "range:", float(x.min()), float(x.max()))
assert x.shape == (3,32,32) and x.min() >= 0 and x.max() <= 1
# Labels are retained by the dataset loader but are not passed into any model.

## 2. Beta-VAE — beta in {1,2,4,10} [2 marks]
The convolutional encoder predicts a mean and log variance, followed by a reparameterized Gaussian sample. A sigmoid decoder reconstructs a 32×32 RGB image. The objective is mean-over-images [sum-of-pixel squared error + beta × sum-of-latent KL]. All beta settings use the same initialization, split and training budget. A common ten-epoch KL warmup reaches the requested beta.

PSNR uses max pixel value 1 and is averaged per image. Reconstructions use the posterior mean for deterministic PSNR. FID instead measures unconditional samples from the standard-normal prior. These are distinct tasks.

## 3. VQ-VAE — K in {512,256,128} [3 marks]
The encoder produces 8×8×64 continuous latents. Nearest-neighbor lookup selects one of K embedding vectors at each spatial location. Straight-through gradients train the encoder; the codebook and commitment losses train the discrete representation. A decoder reconstructs from the chosen embeddings.

After training each VQ-VAE, freeze its encoder/codebook and extract train and validation maps. A masked-convolution PixelCNN models p(code[row,col] | preceding raster codes). Its first type-A mask excludes the current code and later positions. Type-B hidden masks preserve causality. Train by categorical cross-entropy, sample the entire 8×8 map in raster order at temperature 1, then decode the sampled embeddings.

Quality checks include PSNR/MSE, code usage histogram, active fraction, dead entries, perplexity and PixelCNN validation bits/code. A large K is not automatically better; compare perplexity relative to K and inspect sampling quality.

PixelCNN keeps `prior_last.pt` for training recovery and `prior_best.pt` for sampling, selected only by validation cross-entropy.

In [ ]:
from udl.models import BetaVAE, VQVAE, PixelCNN
architecture_examples = {"Beta-VAE": BetaVAE(cfg["latent_dim"]),
                         "VQ-VAE K=256": VQVAE(256,cfg["embedding_dim"],cfg["commitment_cost"]),
                         "PixelCNN K=256": PixelCNN(256)}
for label,model in architecture_examples.items():
    print(label, "parameters:",sum(p.numel() for p in model.parameters()))
    print(model)
del architecture_examples,model

In [ ]:
from udl.pipeline import run_part_a
# Runs/resumes all 4 VAE and 3 VQ-VAE+PixelCNN experiments.
# Logs, latent maps, checkpoints, reconstruction grids and generated samples go to cfg["run_dir"].
part_a_results = run_part_a(cfg)

In [ ]:
from udl.reporting import metrics_table, plot_results, review_flags
results = metrics_table(cfg)
part_a = results[~results.experiment.isin(["wgan", "wgan_gp"])]
columns = ["experiment", "fid", "psnr", "mse", "training_minutes", "sampling_images_per_second",
           "codebook_perplexity", "active_codes", "utilization", "prior_val_bits_per_code"]
display(part_a.reindex(columns=columns).round(4))
for figure in plot_results(cfg):
    if "gan_" not in figure.name: display(Image(filename=str(figure)))
print("Diagnostics:", review_flags(cfg))

In [ ]:
for beta in (1,2,4,10):
    print("beta =", beta, "— original/reconstruction rows alternate; same held-out images")
    display(Image(filename=str(Path(cfg["run_dir"])/f"vae_beta{beta}"/"reconstructions.png")))
for size in (512,256,128):
    print("K =",size)
    path = Path(cfg["run_dir"])/f"vqvae_k{size}"
    display(Image(filename=str(path/"reconstructions.png")))
    maps = torch.load(path/"latent_maps.pt", map_location="cpu", weights_only=True)
    print("Extracted maps:", {key:tuple(value.shape) for key,value in maps.items()})
    print("Code range:",int(maps["train"].min()),int(maps["train"].max()))

In [ ]:
import matplotlib.pyplot as plt
for size in (512,256,128):
    metrics = json.loads((Path(cfg["run_dir"])/f"vqvae_k{size}"/"metrics.json").read_text())
    plt.figure(figsize=(10,2.5)); plt.bar(range(size),metrics["code_counts"])
    plt.title(f"K={size}: full test-set code usage; perplexity={metrics['codebook_perplexity']:.1f}")
    plt.xlabel("Code index"); plt.ylabel("Assignments"); plt.tight_layout(); plt.show()
    history = json.loads((Path(cfg["run_dir"])/f"vqvae_k{size}"/"prior_history.json").read_text())
    plt.figure(figsize=(6,3)); plt.plot([r['epoch'] for r in history],[r['val_bits_per_code'] for r in history])
    plt.title(f"K={size}: PixelCNN validation cross-entropy"); plt.xlabel("Epoch"); plt.ylabel("bits/code"); plt.show()

In [ ]:
# Part C requires 100 examples of beta=1 VAE and K=256 VQ-VAE.
for name in ("vae_beta1","vqvae_k256"):
    print(name,"— 100 unselected unconditional samples")
    display(Image(filename=str(Path(cfg["run_dir"])/name/"samples_100.png")))

## 4. Record the observed conclusions
After the full run, replace this paragraph with your group's interpretation of the printed results: which beta gives the best PSNR/FID; how PSNR changes over epochs; whether larger beta produces the expected tradeoff; which K has the best reconstruction, FID and code utilization; and whether the PixelCNN cross-entropy has converged. Do not state a model ranking until the results support it.

Add the perceptual observations for VAE beta=1 and VQ-VAE K=256 to `observations.json`. The Part C report includes a combined quantitative comparison and the 100-image grids.

## 5. Export
**Save this notebook first.** Run the command below after checking outputs, then proceed to Part B. Use `python finalize_submission.py` only after both saved notebooks, all nine final experiment results, the observations and the JPG are complete.

In [ ]:
import subprocess
if Path("UDL_Group56_PartA.ipynb").is_file():
    subprocess.run([sys.executable,"-m","nbconvert","--to","html","UDL_Group56_PartA.ipynb"],check=True)
else:
    print("Save/download the notebook into the project folder before exporting.")